In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [3]:
import os, zipfile, shutil, glob
from pathlib import Path

ZIP_PATH = "/content/drive/MyDrive/Pictogram_Proyect/0_raw/roboflow_export/Chakana.v5i.yolov8.zip"
DST_DIR  = "/content/drive/MyDrive/Pictogram_Proyect/0_raw/roboflow_unzipped/pictos7"

print("ZIP_PATH:", ZIP_PATH)
print("DST_DIR :", DST_DIR)


ZIP_PATH: /content/drive/MyDrive/Pictogram_Proyect/0_raw/roboflow_export/Chakana.v5i.yolov8.zip
DST_DIR : /content/drive/MyDrive/Pictogram_Proyect/0_raw/roboflow_unzipped/pictos7


In [4]:
shutil.rmtree(DST_DIR, ignore_errors=True)
os.makedirs(DST_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(DST_DIR)

print("✅ Descomprimido en:", DST_DIR)
!find "{DST_DIR}" -maxdepth 3 -type f | head -n 30


✅ Descomprimido en: /content/drive/MyDrive/Pictogram_Proyect/0_raw/roboflow_unzipped/pictos7
/content/drive/MyDrive/Pictogram_Proyect/0_raw/roboflow_unzipped/pictos7/README.dataset.txt
/content/drive/MyDrive/Pictogram_Proyect/0_raw/roboflow_unzipped/pictos7/README.roboflow.txt
/content/drive/MyDrive/Pictogram_Proyect/0_raw/roboflow_unzipped/pictos7/data.yaml
/content/drive/MyDrive/Pictogram_Proyect/0_raw/roboflow_unzipped/pictos7/train/images/001_jpeg.rf.02ce5748f745f27e256e5d515e012471.jpg
/content/drive/MyDrive/Pictogram_Proyect/0_raw/roboflow_unzipped/pictos7/train/images/001_jpeg.rf.245247ffd49eb4512ca1e0fb905e49ac.jpg
/content/drive/MyDrive/Pictogram_Proyect/0_raw/roboflow_unzipped/pictos7/train/images/001_jpeg.rf.66bef46ad37d3b78139dce71df0fe263.jpg
/content/drive/MyDrive/Pictogram_Proyect/0_raw/roboflow_unzipped/pictos7/train/images/001_jpeg.rf.b302c0e36868c1d277bd1e5334e2a222.jpg
/content/drive/MyDrive/Pictogram_Proyect/0_raw/roboflow_unzipped/pictos7/train/images/001_jpeg.rf.c

In [4]:
import glob, os

yamls = glob.glob(os.path.join(DST_DIR, "**", "data.yaml"), recursive=True)
print("YAMLs encontrados:", yamls)

if not yamls:
    raise FileNotFoundError("No encuentro data.yaml dentro del ZIP.")
YAML_PATH = yamls[0]
ROOT = os.path.dirname(YAML_PATH)

print("✅ ROOT del dataset:", ROOT)
print("✅ YAML_PATH:", YAML_PATH)
print("\n--- data.yaml actual ---")
print(open(YAML_PATH, "r", encoding="utf-8").read())


YAMLs encontrados: ['/content/drive/MyDrive/Pictogram_Proyect/0_raw/roboflow_unzipped/pictos7/data.yaml']
✅ ROOT del dataset: /content/drive/MyDrive/Pictogram_Proyect/0_raw/roboflow_unzipped/pictos7
✅ YAML_PATH: /content/drive/MyDrive/Pictogram_Proyect/0_raw/roboflow_unzipped/pictos7/data.yaml

--- data.yaml actual ---
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 7
names: ['chakana', 'condor', 'cuy', 'llama', 'maiz', 'tinaja', 'zampona']

roboflow:
  workspace: cpachecoq
  project: chakana
  version: 5
  license: CC BY 4.0
  url: https://universe.roboflow.com/cpachecoq/chakana/dataset/5


In [5]:
def count_imgs_labels(split):
    img_dir1 = os.path.join(ROOT, "images", split)
    lbl_dir1 = os.path.join(ROOT, "labels", split)

    imgs = []
    for ext in ("*.jpg","*.jpeg","*.png"):
        imgs += glob.glob(os.path.join(img_dir1, ext))
    lbls = glob.glob(os.path.join(lbl_dir1, "*.txt"))

    return len(imgs), len(lbls), img_dir1, lbl_dir1

for sp in ["train","valid","val","test"]:
    imgs, lbls, idd, ldd = count_imgs_labels(sp)
    if os.path.isdir(idd) or os.path.isdir(ldd):
        print(f"{sp}: images={imgs}, labels={lbls} | {idd} | {ldd}")


In [6]:
import os

ROOT = "/content/drive/MyDrive/Pictogram_Proyect/0_raw/roboflow_unzipped/pictos7"
print("ROOT:", ROOT)

for d in ["train", "valid", "test"]:
    print(d, "exists?", os.path.isdir(os.path.join(ROOT, d)))
    print("  images exists?", os.path.isdir(os.path.join(ROOT, d, "images")))
    print("  labels exists?", os.path.isdir(os.path.join(ROOT, d, "labels")))


ROOT: /content/drive/MyDrive/Pictogram_Proyect/0_raw/roboflow_unzipped/pictos7
train exists? True
  images exists? True
  labels exists? True
valid exists? True
  images exists? True
  labels exists? True
test exists? False
  images exists? False
  labels exists? False


In [7]:
import shutil, os

valid_dir = os.path.join(ROOT, "valid")
val_dir   = os.path.join(ROOT, "val")

if os.path.isdir(valid_dir) and not os.path.isdir(val_dir):
    shutil.move(valid_dir, val_dir)
    print("✅ Renombrado: valid -> val")
else:
    print("ℹ️ No se renombró (ya existe val o no existe valid).")


✅ Renombrado: valid -> val


In [8]:
import yaml, os

YAML_PATH = os.path.join(ROOT, "data.yaml")

CLASSES = ["chakana","condor","cuy","llama","maiz","tinaja","zampona"]

data = {
    "path": ROOT,
    "train": "train/images",
    "val": "val/images",     # <- si NO renombraste, usa "valid/images"
    "nc": len(CLASSES),
    "names": CLASSES
}

with open(YAML_PATH, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print("✅ data.yaml actualizado en:", YAML_PATH)
print(open(YAML_PATH, "r").read())


✅ data.yaml actualizado en: /content/drive/MyDrive/Pictogram_Proyect/0_raw/roboflow_unzipped/pictos7/data.yaml
path: /content/drive/MyDrive/Pictogram_Proyect/0_raw/roboflow_unzipped/pictos7
train: train/images
val: val/images
nc: 7
names:
- chakana
- condor
- cuy
- llama
- maiz
- tinaja
- zampona



In [9]:
import glob, os

def split_stats(split):
    img_dir = os.path.join(ROOT, split, "images")
    lbl_dir = os.path.join(ROOT, split, "labels")

    imgs = []
    for ext in ("*.jpg","*.jpeg","*.png"):
        imgs += glob.glob(os.path.join(img_dir, ext))

    # labels existentes
    lbls = glob.glob(os.path.join(lbl_dir, "*.txt"))
    lbl_set = set(os.path.splitext(os.path.basename(x))[0] for x in lbls)

    # imágenes sin label
    missing_lbl = 0
    for p in imgs:
        stem = os.path.splitext(os.path.basename(p))[0]
        if stem not in lbl_set:
            missing_lbl += 1

    # labels vacíos
    empty_lbl = sum(1 for p in lbls if os.path.getsize(p) == 0)

    return len(imgs), len(lbls), empty_lbl, missing_lbl

for sp in ["train","val"]:
    if os.path.isdir(os.path.join(ROOT, sp)):
        imgs, lbls, empty_lbl, missing_lbl = split_stats(sp)
        print(f"{sp}: images={imgs}, labels={lbls}, empty_labels={empty_lbl}, images_without_label={missing_lbl}")


train: images=2535, labels=2535, empty_labels=363, images_without_label=0
val: images=28, labels=28, empty_labels=24, images_without_label=0


In [10]:
import os, glob, shutil

ROOT = "/content/drive/MyDrive/Pictogram_Proyect/0_raw/roboflow_unzipped/pictos7"

val_img_dir = os.path.join(ROOT, "val", "images")
val_lbl_dir = os.path.join(ROOT, "val", "labels")

train_img_dir = os.path.join(ROOT, "train", "images")
train_lbl_dir = os.path.join(ROOT, "train", "labels")

moved = 0
missing_img = 0

for lbl_path in glob.glob(os.path.join(val_lbl_dir, "*.txt")):
    if os.path.getsize(lbl_path) == 0:
        stem = os.path.splitext(os.path.basename(lbl_path))[0]

        # encontrar la imagen correspondiente (cualquier extensión)
        img_path = None
        for ext in (".jpg",".jpeg",".png",".JPG",".JPEG",".PNG"):
            cand = os.path.join(val_img_dir, stem + ext)
            if os.path.exists(cand):
                img_path = cand
                break

        if img_path is None:
            missing_img += 1
            continue

        # mover imagen y label
        shutil.move(img_path, os.path.join(train_img_dir, os.path.basename(img_path)))
        shutil.move(lbl_path, os.path.join(train_lbl_dir, os.path.basename(lbl_path)))
        moved += 1

print("✅ Negativas movidas de val -> train:", moved)
print("⚠️ Labels vacíos sin imagen encontrada:", missing_img)


✅ Negativas movidas de val -> train: 24
⚠️ Labels vacíos sin imagen encontrada: 0


In [11]:
import glob, os

def split_stats(split):
    img_dir = os.path.join(ROOT, split, "images")
    lbl_dir = os.path.join(ROOT, split, "labels")

    imgs = []
    for ext in ("*.jpg","*.jpeg","*.png"):
        imgs += glob.glob(os.path.join(img_dir, ext))

    lbls = glob.glob(os.path.join(lbl_dir, "*.txt"))
    empty_lbl = sum(1 for p in lbls if os.path.getsize(p) == 0)

    return len(imgs), len(lbls), empty_lbl

for sp in ["train","val"]:
    imgs, lbls, empty_lbl = split_stats(sp)
    print(f"{sp}: images={imgs}, labels={lbls}, empty_labels={empty_lbl}")


train: images=2559, labels=2559, empty_labels=387
val: images=4, labels=4, empty_labels=0


In [12]:
!pip install -q albumentations==1.4.3 opencv-python-headless


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.0/137.0 kB 8.8 MB/s eta 0:00:00


In [13]:
import os, shutil

SRC_ROOT = "/content/drive/MyDrive/Pictogram_Proyect/0_raw/roboflow_unzipped/pictos7"
OUT_ROOT = "/content/drive/MyDrive/Pictogram_Proyect/1_dataset/pictos7_aug_x6"

AUG_PER_POS = 6
IMG_SIZE = 768

# recrear salida limpia
shutil.rmtree(OUT_ROOT, ignore_errors=True)
for sp in ["train", "val"]:
    os.makedirs(os.path.join(OUT_ROOT, sp, "images"), exist_ok=True)
    os.makedirs(os.path.join(OUT_ROOT, sp, "labels"), exist_ok=True)

print("SRC_ROOT:", SRC_ROOT)
print("OUT_ROOT:", OUT_ROOT)


SRC_ROOT: /content/drive/MyDrive/Pictogram_Proyect/0_raw/roboflow_unzipped/pictos7
OUT_ROOT: /content/drive/MyDrive/Pictogram_Proyect/1_dataset/pictos7_aug_x6


In [14]:
import glob, shutil, os

def copy_tree(src_dir, dst_dir, patterns):
    os.makedirs(dst_dir, exist_ok=True)
    for pat in patterns:
        for p in glob.glob(os.path.join(src_dir, pat)):
            shutil.copy2(p, os.path.join(dst_dir, os.path.basename(p)))

# copiar train
copy_tree(os.path.join(SRC_ROOT, "train", "images"), os.path.join(OUT_ROOT, "train", "images"),
          ["*.jpg","*.jpeg","*.png","*.JPG","*.JPEG","*.PNG"])
copy_tree(os.path.join(SRC_ROOT, "train", "labels"), os.path.join(OUT_ROOT, "train", "labels"),
          ["*.txt"])

# copiar val
copy_tree(os.path.join(SRC_ROOT, "val", "images"), os.path.join(OUT_ROOT, "val", "images"),
          ["*.jpg","*.jpeg","*.png","*.JPG","*.JPEG","*.PNG"])
copy_tree(os.path.join(SRC_ROOT, "val", "labels"), os.path.join(OUT_ROOT, "val", "labels"),
          ["*.txt"])

print("✅ Copiado train+val al OUT_ROOT (sin augment todavía)")


✅ Copiado train+val al OUT_ROOT (sin augment todavía)


In [15]:
import cv2, numpy as np, albumentations as A
import os, glob
from pathlib import Path

train_img_dir = os.path.join(SRC_ROOT, "train", "images")
train_lbl_dir = os.path.join(SRC_ROOT, "train", "labels")

out_train_img = os.path.join(OUT_ROOT, "train", "images")
out_train_lbl = os.path.join(OUT_ROOT, "train", "labels")

def list_imgs(folder):
    imgs=[]
    for ext in ("*.jpg","*.jpeg","*.png","*.JPG","*.JPEG","*.PNG"):
        imgs += glob.glob(os.path.join(folder, ext))
    return sorted(imgs)

def read_yolo(txt_path):
    bboxes, cls = [], []
    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            p = line.strip().split()
            if len(p) != 5:
                continue
            cls.append(int(float(p[0])))
            bboxes.append([float(p[1]), float(p[2]), float(p[3]), float(p[4])])
    return bboxes, cls

def write_yolo(txt_path, bboxes, cls):
    with open(txt_path, "w", encoding="utf-8") as f:
        for c, bb in zip(cls, bboxes):
            x,y,w,h = bb
            f.write(f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")

def clamp_yolo(bb):
    x,y,w,h = bb
    w = max(1e-6, min(1.0, w)); h = max(1e-6, min(1.0, h))
    x = max(0.0, min(1.0, x)); y = max(0.0, min(1.0, y))
    x1 = max(0.0, x - w/2); y1 = max(0.0, y - h/2)
    x2 = min(1.0, x + w/2); y2 = min(1.0, y + h/2)
    w2 = max(1e-6, x2-x1); h2 = max(1e-6, y2-y1)
    return [x1 + w2/2, y1 + h2/2, w2, h2]

transform = A.Compose(
    [
        A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE,
                      border_mode=cv2.BORDER_CONSTANT, value=(0,0,0), p=1.0),

        # simular distancia (pequeño) pero sin destruir el objeto
        A.RandomScale(scale_limit=(-0.65, -0.15), interpolation=cv2.INTER_LINEAR, p=0.55),

        A.Affine(
            scale=(0.90, 1.12),
            translate_percent=(0.06, 0.06),
            rotate=(-25, 25),
            shear=(-7, 7),
            cval=(0,0,0),
            p=0.90
        ),

        A.Resize(IMG_SIZE, IMG_SIZE),

        A.RandomBrightnessContrast(0.20, 0.20, p=0.60),
        A.RandomGamma(gamma_limit=(85, 125), p=0.35),
        A.HueSaturationValue(p=0.25),
        A.GaussianBlur(blur_limit=(3,5), p=0.25),
    ],
    bbox_params=A.BboxParams(
        format="yolo",
        label_fields=["class_labels"],
        min_visibility=0.35
    )
)

imgs = list_imgs(train_img_dir)
made = 0
skipped_neg = 0

for img_path in imgs:
    stem = Path(img_path).stem
    lbl_path = os.path.join(train_lbl_dir, stem + ".txt")

    # solo positivas: label existe y NO está vacío
    if (not os.path.exists(lbl_path)) or os.path.getsize(lbl_path) == 0:
        skipped_neg += 1
        continue

    bgr = cv2.imread(img_path)
    if bgr is None:
        continue
    img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    bboxes, cls = read_yolo(lbl_path)
    if not bboxes:
        continue

    for i in range(AUG_PER_POS):
        try:
            aug = transform(image=img, bboxes=bboxes, class_labels=cls)
            aug_img = aug["image"]
            aug_boxes = [clamp_yolo(list(bb)) for bb in aug["bboxes"]]
            aug_cls = list(aug["class_labels"])

            out_img_name = f"{stem}_aug_{i:02d}.jpg"
            out_lbl_name = f"{stem}_aug_{i:02d}.txt"

            cv2.imwrite(os.path.join(out_train_img, out_img_name),
                        cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR))
            write_yolo(os.path.join(out_train_lbl, out_lbl_name), aug_boxes, aug_cls)

            made += 1
        except Exception:
            continue

print("✅ Augment listo")
print("   Generadas:", made)
print("   Saltadas (negativas):", skipped_neg)


✅ Augment listo
   Generadas: 13032
   Saltadas (negativas): 387


In [16]:
import yaml, os

CLASSES = ["chakana","condor","cuy","llama","maiz","tinaja","zampona"]
YAML_OUT = os.path.join(OUT_ROOT, "data.yaml")

data = {
    "path": OUT_ROOT,
    "train": "train/images",
    "val": "val/images",
    "nc": len(CLASSES),
    "names": CLASSES
}

with open(YAML_OUT, "w", encoding="utf-8") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print("✅ data.yaml augmentado:", YAML_OUT)
print(open(YAML_OUT, "r").read())


✅ data.yaml augmentado: /content/drive/MyDrive/Pictogram_Proyect/1_dataset/pictos7_aug_x6/data.yaml
path: /content/drive/MyDrive/Pictogram_Proyect/1_dataset/pictos7_aug_x6
train: train/images
val: val/images
nc: 7
names:
- chakana
- condor
- cuy
- llama
- maiz
- tinaja
- zampona



In [17]:
import glob, os

def count_split(root, split):
    img_dir = os.path.join(root, split, "images")
    lbl_dir = os.path.join(root, split, "labels")
    imgs = sum(len(glob.glob(os.path.join(img_dir, ext))) for ext in ["*.jpg","*.jpeg","*.png","*.JPG","*.JPEG","*.PNG"])
    lbls = len(glob.glob(os.path.join(lbl_dir, "*.txt")))
    empty = sum(1 for p in glob.glob(os.path.join(lbl_dir, "*.txt")) if os.path.getsize(p) == 0)
    return imgs, lbls, empty

for sp in ["train","val"]:
    imgs, lbls, empty = count_split(OUT_ROOT, sp)
    print(f"{sp}: images={imgs}, labels={lbls}, empty_labels={empty}")


train: images=15591, labels=15591, empty_labels=387
val: images=4, labels=4, empty_labels=0


In [2]:
import os
import random
import shutil

DATASET_ROOT = "/content/drive/MyDrive/Pictogram_Proyect/1_dataset/pictos7_aug_x6"

train_img = os.path.join(DATASET_ROOT, "train/images")
train_lbl = os.path.join(DATASET_ROOT, "train/labels")

val_img = os.path.join(DATASET_ROOT, "val/images")
val_lbl = os.path.join(DATASET_ROOT, "val/labels")

os.makedirs(val_img, exist_ok=True)
os.makedirs(val_lbl, exist_ok=True)

images = [f for f in os.listdir(train_img) if f.endswith((".jpg",".png",".jpeg"))]

# mover 10%
num_to_move = int(len(images) * 0.01)
selected = random.sample(images, num_to_move)

for img in selected:
    lbl = img.rsplit(".",1)[0] + ".txt"

    shutil.move(os.path.join(train_img, img), os.path.join(val_img, img))

    if os.path.exists(os.path.join(train_lbl, lbl)):
        shutil.move(os.path.join(train_lbl, lbl), os.path.join(val_lbl, lbl))
    else:
        # crear label vacío si no existe (negativa)
        open(os.path.join(val_lbl, lbl), "w").close()

print("✅ Imágenes movidas a val:", num_to_move)


✅ Imágenes movidas a val: 155
